In [0]:

import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql.functions import lit, col

In [0]:
api_key = dbutils.fs.head('dbfs:/Volumes/api_football/default/secret/api_key.txt')
#print(api_key)

In [0]:
url = (
    f'https://apiv3.apifootball.com/'
    f'?action=get_events'
    f'&from=2026-01-01'
    f'&to=2026-12-31'
    f'&league_id=99' #brasil
    f'&APIkey={api_key}'
)

# Request
response = requests.get(url)

# Status
print(response.status_code)

# JSON
data = response.json()

# DataFrame
df_raw = pd.DataFrame(data)

# Convert to Spark DataFrame
df_raw = spark.createDataFrame(df_raw)

dh_processing_br = datetime.now() - timedelta(hours=3)
df_raw = df_raw.withColumn('dh_processing_br', lit(dh_processing_br))
#df_raw.createOrReplaceTempView('vw_raw')



In [0]:
# Drop columns with VOID type that Delta cannot write
df_clean = df_raw.withColumn("lineup", 
    col("lineup").withField("away", 
        col("lineup.away").dropFields("missing_players"))
    .withField("home", 
        col("lineup.home").dropFields("missing_players")))


In [0]:
df_clean.write.mode("append").saveAsTable('api_football.bronze.matches_raw')